# 03: Serve the generator with Ray Serve

This notebook deploys the interactive application: pick a real drive, choose a condition, and Cosmos Transfer 2.5 generates it on a GPU.

* **`Painter`**: 1 GPU per replica, `max_ongoing_requests=1`. Wraps the same actor class as the batch pipeline, so the model loads once per replica.
* **`PaintAPI`**: a FastAPI front end on CPU. It serves the page and a job API: `POST /api/paint` returns job IDs at once and the browser polls for progress.
* **Scene affinity**: scene IDs are passed as multiplexed model IDs, so Serve sends repeat requests for a drive to a replica that has already encoded it.

With `USE_GPUS = False` the application runs in **replay mode**: a CPU `ReplayPainter` returns stored guardrails-on outputs for the preset conditions, so you can try the whole application, including routing and the UI, without GPUs.

In [1]:
# Setup
USE_GPUS = False   # False: stored results on CPU. True: generate live on GPUs (template image + weights).

import os, sys
sys.path.insert(0, os.getcwd())
import pandas as pd
import notebook_helpers as nh

pd.set_option("display.max_colwidth", 80)
nh.check_setup(USE_GPUS)

COSMOS_SHARED = /mnt/user_storage/cosmos-video-augmentation  (99 videos in the bundle)
Mode: stored results on CPU (set USE_GPUS = True to generate)


```python
@serve.deployment(max_ongoing_requests=1, ray_actor_options={"num_gpus": 1, "num_cpus": 4})
class Painter:
    def __init__(self, settings):
        self.engine = CosmosActor(idx, settings)          # same engine as the batch pipeline

    @serve.multiplexed(max_num_models_per_replica=12)
    async def _scene(self, scene_id): return scene_id

    async def paint(self, req):
        await self._scene(serve.get_multiplexed_model_id())
        return await asyncio.to_thread(self.engine.generate, req)

# In PaintAPI: route each job to a replica that already holds its drive
self.painter.options(multiplexed_model_id=job["scene_id"]).paint.remote(req)
```

In [2]:
import logging
from ray import serve
from offroad_ray.ray_env import init_ray
from offroad_ray.serve.app import build_app

logging.getLogger("ray.serve").setLevel(logging.WARNING)

ROUTE = os.environ.get("PAINT_ROUTE", "/")
APP_NAME = os.environ.get("PAINT_APP", "paint")
init_ray(log_to_driver=False)   # keep replica request logs out of the notebook
serve.run(build_app(use_gpus=USE_GPUS, replicas=8 if USE_GPUS else 4), name=APP_NAME, route_prefix=ROUTE)
print(serve.status().applications[APP_NAME].status)

ApplicationStatus.RUNNING


## Call the API

Send one preset condition to three drives, then poll the jobs until they finish. In GPU mode each clip takes about 80 seconds and all three run in parallel; in replay mode they return after a few seconds.

In [3]:
import time, requests

BASE = "http://localhost:8000" + ROUTE.rstrip("/")
info = requests.get(f"{BASE}/api/scenes").json()
print("mode:", info["mode"], "| replicas:", info["replicas"], "| scenes:", len(info["scenes"]))
preset = next(p for p in info["presets"] if p[0].lower().startswith("golden"))
scene_ids = [s["scene_id"] for s in info["scenes"][:3]]

jobs = requests.post(f"{BASE}/api/paint", json={"prompt": preset[2], "scenes": scene_ids}).json()["jobs"]
t0 = time.time()
while True:
    states = [requests.get(f"{BASE}/api/jobs/{j['job_id']}").json() for j in jobs]
    if all(s["status"] in ("done", "failed") for s in states):
        break
    time.sleep(2)
print(f"finished in {time.time() - t0:.1f} s")
pd.DataFrame(states)[["scene_id", "status", "cached", "seconds", "error"]]

mode: replay | replicas: 4 | scenes: 9


finished in 4.1 s


,scene_id,status,cached,seconds,error
0,fb9d3bafac,done,False,3.04,None
1,dd5a2ee56e,done,False,3.04,None
2,5d20270fcc,done,False,3.04,None


The same request again returns at once: outputs are stored by a hash of the drive, prompt and seed.

In [4]:
again = requests.post(f"{BASE}/api/paint", json={"prompt": preset[2], "scenes": scene_ids}).json()["jobs"]
pd.DataFrame(again)[["scene_id", "status", "cached"]]

,scene_id,status,cached
0,fb9d3bafac,done,True
1,dd5a2ee56e,done,True
2,5d20270fcc,done,True


## Open the page

In a Workspace, open the Workspace URL with `?port=8000` added. If your browser reports an invalid referer, paste the URL into a new tab instead of clicking a link to it.

The page walks a first-time visitor through three steps (pick a drive, pick a condition, generate), shows each GPU's progress and lays out the results with a slider that compares the real and generated clips.

![The live application](docs/images/app.gif)

## Share it as an Anyscale Service

```bash
anyscale service deploy -f jobs/service.yaml     # prints a URL to share
anyscale service terminate -n cosmos-video-augmentation
```

The Service runs on its own cluster, with 8 A100 replicas in GPU mode. It keeps its GPUs until you terminate it. Anyone with the URL can submit generation requests, so keep the guardrails on and share the URL only with your audience.

## Clean up

In [5]:
serve.delete(APP_NAME)
print("application deleted")

application deleted
